# Exercise 1 — The Planck PSZ2 Catalogue and the Y–M Scaling Relation

**Course topics covered:**
- Reading and inspecting FITS binary tables with `astropy.io.fits`.
- Visualising survey sky coverage in Galactic coordinates.
- Identifying and handling missing or flagged data in astronomical catalogues.
- Computing the observed cluster mass function and its Poisson uncertainties.
- Fitting power-law models with `scipy.optimize.least_squares` and χ² minimisation.
- Bayesian parameter estimation with a custom Metropolis–Hastings sampler.

**Reference papers:**
- Planck Collaboration, [Ade, P. A. R., et al. 2016, *A&A* **594**, A27](https://arxiv.org/abs/1502.01598) — the second Planck catalogue of SZ sources (PSZ2); the dataset used in this exercise.
- [Kravtsov, A. V., Vikhlinin, A., & Nagai, D. 2006, *ApJ* **650**, 128](https://arxiv.org/abs/astro-ph/0602117) — derivation of the Y–M scaling relation from simulations; discusses the self-similar expectation Y ∝ M^(5/3).

**Dataset:** `planck_sz_catalogue.fit` — 1653 galaxy cluster candidates detected by the Planck satellite via the Sunyaev–Zel'dovich (SZ) effect. Key columns: `GLON`/`GLAT` (Galactic coordinates, deg); `SNR` (detection signal-to-noise); `q_neural` (neural-network quality score, 0–1); `COSMO` (flag for the cosmological sub-sample); `Z` (spectroscopic redshift; −1 if unavailable); `Y5R500`/`e_Y5R500` (integrated Compton-y within 5R₅₀₀, in units of 10⁻³ arcmin², with uncertainty); `MSZ`/`E_MSZ` (mass M₅₀₀ inferred from the Y–M scaling relation, in units of 10¹⁴ M☉, with upper uncertainty).

**Scientific scenario.** The Sunyaev–Zel'dovich (SZ) effect is a spectral distortion of the cosmic microwave background (CMB) caused by inverse Compton scattering of CMB photons off the hot electrons of the intracluster medium. The integrated Compton-y parameter Y₅R₅₀₀ is proportional to the total thermal energy of the gas and is tightly correlated with the cluster mass M through a power-law scaling relation, Y ∝ M^α. Calibrating this Y–M relation and measuring the cluster mass function are fundamental steps for using SZ-selected cluster catalogues as cosmological probes of the growth of large-scale structure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.io.fits as fits
import astropy.units as u
from astropy.cosmology import Planck15
from scipy.optimize import least_squares, minimize

# Add any additional imports you need

## Task 0 — Angular distribution on the sky

Before loading individual cluster properties, explore the sky coverage of the survey.

- Plot the cluster positions in Galactic coordinates (Aitoff projection). Is the distribution isotropic?
- Plot the distribution of Galactic latitudes |b| and compare it with the expectation for a uniform distribution on the sphere, P(|b|) d|b| ∝ cos(|b|) d|b|. What does the deficit at low |b| reveal about how the catalogue was constructed?
- What fraction of the sky is excluded by the Galactic mask? Can you estimate it from the plot?
- How does the spatial distribution of the cosmological sub-sample (`COSMO` = 1) differ from that of the full catalogue?

In [ ]:
# Load the FITS file with fits.open() and access the binary table in extension [1].
# Extract the GLON, GLAT, and COSMO columns.

# YOUR CODE HERE

# The Aitoff projection requires longitude in radians in the range [-π, π].
# GLON is in [0, 360]: shift values > 180 by −360 before converting to radians.

# YOUR CODE HERE

# Use fig.add_subplot(111, projection="aitoff") for the Aitoff projection.
# Plot the full catalogue and the cosmological sub-sample with different colours.

# YOUR CODE HERE

In [ ]:
# Compute the histogram of |b| in 5-deg bins over [0, 90].

# For a uniform distribution on the sphere: dN/d|b| ∝ cos(|b|).
# Normalise the expected counts so the total equals the number of clusters:
#   cos_b = cos(b_centers_rad) * delta_b_rad
#   n_iso = N_total * cos_b / cos_b.sum()

# Plot the observed histogram (bar) and the isotropic expectation (dashed line).

# YOUR CODE HERE

## Task 1 — Explore the catalogue columns

Inspect the available observables and the basic statistics of the detection quality indicators.

- What columns are available? Which ones encode detection quality, and how do they differ?
- Plot the distribution of detection SNR. What is the minimum SNR in the catalogue, and why was this threshold chosen?
- How does the neural-network quality score `q_neural` correlate with SNR? What does a low `q_neural` indicate?
- How many clusters belong to the cosmological sub-sample (`COSMO` = 1), and what selection criteria define it?

In [ ]:
# Print the total number of cluster candidates and the size of the cosmological sub-sample.
# List all available columns using psz2.columns.info().

# YOUR CODE HERE

In [ ]:
# Extract the SNR and q_neural columns.
# Plot side by side:
#   Left : histogram of SNR.
#   Right: scatter plot of SNR vs q_neural, colour-coded by the COSMO flag.

# YOUR CODE HERE

## Task 2 — Redshift distribution

Entries without a confirmed optical/X-ray counterpart are flagged with Z = −1 in the catalogue.

- What fraction of the candidates has a measured spectroscopic redshift?
- What is the redshift range of the confirmed clusters?
- Why might some SZ detections lack a redshift measurement? Is the unconfirmed fraction uniform across the sky?
- How does the redshift distribution compare with the expectation from a survey with a roughly mass-limited selection (the comoving volume grows as z² at low z, but the mass threshold also rises with z)?

In [ ]:
# Extract the redshift column Z.
# Create a boolean mask has_z = (Z > 0) to select confirmed clusters.
# Print the fraction with measured redshift and the redshift range.
# Plot the histogram of Z for confirmed clusters only.

# YOUR CODE HERE

## Task 3 — The Y–M scaling relation

The integrated Compton-y parameter Y₅R₅₀₀ (column unit: 10⁻³ arcmin²) and the SZ mass M_SZ are expected to follow a power-law scaling relation M_SZ = A · Y₅R₅₀₀^B.

- Plot M_SZ as a function of Y₅R₅₀₀ with error bars on a log–log scale. Does the relation look consistent with a power law?
- Why do some entries have `NaN` for both Y₅R₅₀₀ and M_SZ? What does the redshift of those entries confirm?
- The self-similar prediction is Y ∝ M^(5/3), i.e., B = 3/5 = 0.6 for the inverse relation M(Y). Does the scatter in the plot seem consistent with this slope?
- Note that M_SZ was itself derived from Y₅R₅₀₀ using a reference scaling relation. What does this imply for the fit you will perform in Task 4?
- Optional: colour-code the data points by redshift.

In [ ]:
# Build a boolean mask mask_ym selecting entries where both Y5R500 and MSZ are finite
# (use np.isfinite). Print how many clusters pass the mask and inspect the Z values
# of the excluded entries.

# YOUR CODE HERE

# Plot MSZ vs Y5R500 on a log-log scale with plt.errorbar.
# Use e_Y5R500 for x-errors and E_MSZ for y-errors.

# YOUR CODE HERE

## Task 4 — Fitting the scaling relation

Fit the power-law model M_SZ = A · Y₅R₅₀₀^B using two different optimisation strategies and compare their results.

- **Least squares** (`scipy.optimize.least_squares`): minimise the sum of squared residuals, ignoring measurement uncertainties on M_SZ.
- **χ² minimisation** (`scipy.optimize.minimize`): minimise the weighted sum χ² = Σ [(M_SZ − model)² / σ²_M], accounting for the reported upper uncertainties `E_MSZ`.
- Do both methods converge to the same solution? When would you expect them to differ significantly?
- Visualise the χ² landscape as a function of A and B. What does it reveal about parameter degeneracy?
- Optional: estimate parameter uncertainties from the Δχ² = 1 (68% C.L.) contour.

In [ ]:
def power_law(x, A, B):
    # YOUR CODE HERE
    pass


def chi2(params, x, y, yerr):
    # Compute the weighted sum of squared residuals:
    #   chi2 = sum( (y - model)^2 / yerr^2 )
    # YOUR CODE HERE
    pass


# Extract the data arrays using mask_ym.
x    = ...  # Y5R500
y    = ...  # MSZ
yerr = ...  # E_MSZ

In [ ]:
# Define a residuals function: residuals(params) = power_law(x, A, B) - y
# Pass it to least_squares with x0 = [1.0, 1.0].
# Store the best-fit parameters as A_ls, B_ls and print them.

# YOUR CODE HERE

In [ ]:
# Minimise chi2(params, x, y, yerr) with scipy.optimize.minimize (method="Powell").
# Use the least-squares result as the starting point.
# Store the best-fit parameters as A_chi2, B_chi2 and print them.

# YOUR CODE HERE

In [ ]:
# Compute the chi2 landscape on a 100×100 grid:
#   A_values = np.linspace(0.5, 10.0, 100)
#   B_values = np.linspace(-0.20, 0.05, 100)
#
# Evaluate chi2([A, B], x, y, yerr) for each (A, B) pair.
# Reshape to (100, 100) where Z_chi2[i, j] = chi2 at (A_values[i], B_values[j]).
# Use imshow with np.log10(Z_chi2).T, origin="lower", and the correct extent.
# Mark both best-fit points.

A_values = np.linspace(0.5, 10.0, 100)
B_values = np.linspace(-0.20, 0.05, 100)

# YOUR CODE HERE

In [ ]:
# Overplot both power-law fits on the data.
# Use np.logspace to build a smooth x_fit array.
# Include the fit parameters in the legend labels.

# YOUR CODE HERE

## Task 5 — Cluster mass function

The cluster mass function (CMF) describes the number density of clusters per unit comoving volume per unit mass interval: dn/dM (or equivalently dn/d log₁₀ M). It is one of the most powerful cosmological observables.

- Plot the observed differential count dN/d log₁₀ M for all clusters with a valid mass estimate. Add Poisson error bars (√N per bin).
- The raw counts are biased by Malmquist-type selection: massive clusters are visible to larger distances, so the observed dN/dM does not equal the intrinsic dn/dM. To reduce this bias, select a **volume-limited** subsample (e.g., z < 0.3) and compute a rough estimate of the volume-normalised CMF by dividing by the survey volume V_survey = f_sky × V_comoving(z_max), using f_sky ≈ 0.83.
- How does the CMF compare qualitatively with the expectation of an exponential decline at high masses (Press–Schechter / Tinker form)?
- What are the main sources of systematic uncertainty in this estimate (completeness, mass bias, redshift incompleteness)?

In [ ]:
# Select clusters with a finite MSZ value (mask_m = np.isfinite(...)).
# Bin log10(MSZ) with np.histogram; use ~18 bins.
# Compute dN/d(log10 M) = n / dlogm, with Poisson errors sqrt(n) / dlogm.
# Plot on a log-log scale.

# YOUR CODE HERE

In [ ]:
z_max = 0.3
f_sky = 0.83  # Planck sky coverage fraction outside the Galactic mask

# Select the volume-limited subsample: has_z & mask_m & (z > 0) & (z < z_max).
# Compute the comoving survey volume:
#   V_survey = f_sky * Planck15.comoving_volume(z_max).to(u.Mpc**3).value
# Bin log10(MSZ) and compute the volume-normalised CMF:
#   phi     = n / (V_survey * dlogm)
#   phi_err = sqrt(n) / (V_survey * dlogm)
# Plot on a log-log scale.

# YOUR CODE HERE

## Optional — Bayesian inference for the Y–M scaling relation

Implement a custom Metropolis–Hastings (MH) sampler and use it to explore the posterior of the scaling-relation parameters, first for the two-parameter model and then for an extended model that includes intrinsic scatter.

In [ ]:
def metropolis_hastings(log_posterior, theta0, proposal_std, n_steps, seed=42):
    """
    Metropolis–Hastings sampler with a diagonal Gaussian proposal.

    Parameters
    ----------
    log_posterior : callable(theta) -> float
    theta0        : array-like, starting point
    proposal_std  : array-like, per-parameter proposal standard deviations
    n_steps       : int, number of MCMC steps
    seed          : int, random seed

    Returns
    -------
    samples         : ndarray, shape (n_steps + 1, n_params)
    acceptance_rate : float
    """
    # 1. Initialise: allocate samples array (n_steps+1, n_params), set samples[0] = theta0.
    # 2. At each step i:
    #      a. Draw proposal = current + rng.normal(0, proposal_std, n_params)
    #      b. Compute log_alpha = log_posterior(proposal) - log_posterior(current)
    #      c. Accept if log(rng.uniform()) < log_alpha; otherwise keep current.
    #      d. Store current in samples[i+1].
    # 3. Return samples and n_accepted / n_steps.

    # YOUR CODE HERE
    pass

### Subtask 1 — Two-parameter fit: sampling the posterior P(A, B | data)

Assume a Gaussian likelihood with known measurement uncertainties and flat priors on A > 0 and B:

ln P(A, B | data) = −χ²(A, B) / 2

- Run the chain for N = 50 000 steps starting from the χ² minimum. What acceptance rate do you obtain, and how does it depend on the proposal width?
- Discard the burn-in period and plot the traces for A and B. Do they look stationary?
- Plot the 2D posterior and the marginalised 1D posteriors. How do the posterior means and standard deviations compare with the χ² minimum?
- Compare the 68% credible interval from the MH chain with the Δχ² = 1 contour from the landscape plot in Task 4.

In [ ]:
# Define log_post_2p(params):
#   - Return -np.inf if A <= 0 (prior boundary).
#   - Otherwise return -0.5 * chi2(params, x, y, yerr).

# YOUR CODE HERE

# Run metropolis_hastings with:
#   theta0       = chi2_result.x
#   proposal_std = [0.01, 0.0005]   (tune to reach ~25% acceptance)
#   n_steps      = 50_000
# Discard the first burn_in = 5000 steps.
# Print the acceptance rate and posterior mean ± std for A and B.

burn_in = 5_000

# YOUR CODE HERE

In [ ]:
# In a 2×2 figure plot:
#   Top-left    : trace of A (thinned by 10) with the chi2 minimum as a reference line.
#   Bottom-left : trace of B (thinned by 10) with the chi2 minimum as a reference line.
#   Top-right   : marginalised posterior histogram of A.
#   Bottom-right: marginalised posterior histogram of B.
#
# Then in a separate figure, plot the 2D posterior with plt.hist2d(A_samp, B_samp).

# YOUR CODE HERE

### Subtask 2 — Three-parameter fit: adding intrinsic scatter

The χ² fit in Subtask 1 assumes that all scatter around the scaling relation is due to measurement noise σ_M. In practice, physical processes (merger history, gas physics, projection effects) introduce an **intrinsic scatter** σ_int that adds in quadrature to the measurement uncertainty. The correct likelihood per cluster is then:

ln L_i = −½ [ (M_i − A Y_i^B)² / (σ_M,i² + σ_int²) + ln(σ_M,i² + σ_int²) ]

This introduces a third free parameter σ_int ≥ 0.

- Extend the log-posterior to θ = (A, B, σ_int) and re-run the MH sampler.
- What value of σ_int does the chain converge to? Is the intrinsic scatter comparable to, larger, or smaller than the typical measurement uncertainty?
- How do the marginalised posteriors for A and B change when σ_int is included? Do the uncertainties on A and B increase?
- What is the physical interpretation of σ_int in the context of galaxy cluster scaling relations?

In [ ]:
# Define log_post_3p(params) with params = (A, B, sigma_int):
#   - Return -np.inf if A <= 0 or sigma_int < 0.
#   - Compute total_var = yerr**2 + sigma_int**2
#   - Return -0.5 * sum( residuals**2 / total_var + log(total_var) )
#     where residuals = y - power_law(x, A, B).
#     Note: the log(total_var) term is essential — it penalises large sigma_int.

# YOUR CODE HERE

# Initialise: theta0_3p = [A_chi2, B_chi2, np.median(yerr)]
# Run metropolis_hastings for 50 000 steps.
# Discard burn-in and print the posterior mean ± std for all three parameters.
# Compare sigma_int with np.median(yerr) to assess its relative importance.

# YOUR CODE HERE

In [ ]:
# In a 1×3 figure, plot the marginalised posteriors for A, B, and sigma_int.
# For A and B, overlay the 2-parameter posterior (from Subtask 1) to show
# how the uncertainties change when sigma_int is freed.
#
# In a separate 1×2 figure, plot 2D histograms:
#   Left : A vs sigma_int
#   Right: B vs sigma_int

# YOUR CODE HERE